# Vision app

In [1]:
%matplotlib widget
from dorna_vision import Detection_app
x = Detection_app()
#model/collection_tube_kp.pkl
#{"tube":{"head": [0, -40, 0], "tail": [0, -20, 0]}}

# Eye in hand calibration

In [17]:
from dorna2 import Dorna
from camera import Camera
from dorna_vision import Detection
import time
import json
#####################
import config

pxl_list = [[456, 236], [515, 90], [325, 121], [561, 332], [364, 358]]
pxl = pxl_list[0]
rvec_base = [180, 0, 0]
ip = config.robot["ip"]
joint_img = config.tray_to_well_plate["end"]["loc"][0]+config.tray_to_well_plate["end"]["loc"][1]
tool = config.suction_gripper["tool"]
preset = {
        "camera_mount":{
            "type": "dorna_ta_j4",
            "ej": [0 ,0, 0, 0, 0, 0, 0, 0],
            "T": [46.5174596+1+1+0+4, 32.0776662-3+1-0-1.5, -4.24772615-3, -0.27547989, 0.27691881, 89.6939516],
        },
}
sim=1
speed = 0.05
###### initialization
# robot
robot = Dorna()
robot.connect(ip)

# camera
camera = Camera()
camera.connect()
Detection(robot=robot, camera=camera, **config.det_preset["collection_tube_kp"])
# detection
d = Detection(robot=robot, camera=camera, **preset)
######
# go imaging
robot.go(joint=joint_img)
time.sleep(0.5)

# run detection
d.run()

# get xyz
xyz = d.xyz(pxl)

# go xyz
tvec = xyz+rvec_base
retval = robot.go(pose=tvec, tool=tool, speed=speed, sim=sim)
for r in retval:
    print(json.dumps(r))

robot.close()
camera.close()
d.close()

{"cmd": "jmove", "rel": 0, "accel": 100.0, "jerk": 200.0, "cont": 0, "corner": 50, "j0": -108.7543308888869, "j1": 17.171870179493716, "j2": -101.34596743176473, "j3": -10.943655513023828, "j4": -4.495310125655465, "j5": -96.59312130037239}


# Calibration with joints

In [1]:
from dorna2 import Dorna
from dorna2 import pose as dorna_pose
import numpy as np
import config

# rail config
frame_in_world = [0, 0, 0, 0, 0, 0]
base_in_world = [127.5, 12.5, 0, 0, 0, 0]
aux_dir = [[0, 0, 0], [0, 0, 0]]

# measured
m_joint_rec = {"cmd":"jmove","rel":0,"j0":-68.620605,"j1":40.36377,"j2":-116.740723,"j3":-2.15332,"j4":-14.0625,"j5":113.730469}
m_aux = config.well_plate["aux"]

# target[
target_aux = config.well_plate["aux"]
target_frame = config.well_plate["frame"]
tool = config.two_finger_gripper["tool_wo_tube"]
target_pose = config.well_plate["tube"]["a1"]
target_pose[2] -= 4


####### measured
m_joint = [m_joint_rec[k] for k in ["j"+str(i) for i in range(6)]]
robot = Dorna()
robot.kinematic.set_tcp_xyzabc(tool)
robot.kinematic.fw(m_joint)
m_xyzabc = robot.kinematic.fw(m_joint)
m_xyzabc_world = dorna_pose.robot_to_frame(robot.kinematic.fw(m_joint), aux=m_aux, aux_dir=aux_dir, base_in_world=base_in_world, frame_in_world = frame_in_world)
print(m_xyzabc_world)

####### object
target_xyzabc_world = dorna_pose.transform_pose(target_pose, from_frame=target_frame, to_frame=[0,0,0,0,0,0])
print(target_xyzabc_world)

####### recalculate o_frame
# ── 2) Build 4×4 transforms for world and local poses ────────────────────
T_world = np.array(dorna_pose.xyzabc_to_T(m_xyzabc_world))
T_local = np.array(dorna_pose.xyzabc_to_T(target_pose))

# ── 3) Solve for the object frame:  T_frame = T_world @ inv(T_local) ─────
T_frame = T_world @ np.linalg.inv(T_local)
target_frame_np = dorna_pose.T_to_xyzabc(T_frame)
target_frame_new = [float(x) for x in target_frame_np]
print("####### target_frame_new ###########")
print(target_frame_new)

#### recalculate target_pose
target_pose_new = list(map( float, dorna_pose.transform_pose( m_xyzabc_world, from_frame=[0, 0, 0, 0, 0, 0], to_frame=target_frame)))
print("####### target_pose_new ###########")
print(target_pose_new)

[271.46941613386014, -228.75568585382277, 83.14760291231087, 1.0105966612608028, -0.32651207748820477, 179.73660117474088]
[269.60220024396745, -228.7836177706273, 82.5419139445724, 0.857507918964842, -0.28135648736373425, 179.71610956564567]
####### target_frame_new ###########
[237.79567976233722, -281.8059603358065, 0.8261422767794073, 1.0105966612737578, -0.32651207749238487, 179.73660117704614]
####### target_pose_new ###########
[-34.36618692274976, -53.53901928337075, 83.12337407003739, -0.02842595548114894, -0.09764048643474471, 0.021346196000313757]


# Rotation

In [3]:
from dorna2 import pose

abc = [0, 0, 0]

abc = pose.rotate_abc(abc, axis=[1, 0, 0], angle=180, local=True)
abc = pose.rotate_abc(abc, axis=[0, 0, 1], angle=45, local=True)
abc

[166.2983158520316, -68.88301782571617, 4.217729064991604e-15]

# Go

In [2]:
from dorna2 import Dorna
import time
freedom = {"num":50, "range":[0.5, 0.5, 0.5], "early_exit":True}
robot = Dorna()
current_joint =  [80.112305, 68.862305, -117.685547, -0.021973, -41.176758, 35.134277, 388.74375]
pose =  [-300.4584516808835, 186.73230510526315, 250.1425720414547, -159.00911604446466, 65.86373240657448, -23.19692136401907, 390.35]

final_joint = [141.745605, 48.88916, -80.002441, -10.700684, -72.355957, -168.815918, 390.35625]


#robot.kinematic.inv(pose[0:6], current_joint[0:6], False, freedom=freedom)
robot.go(pose=pose, current_joint=current_joint, motion="lmove", freedom= freedom, cont=0, corner=50, timeout=-1, sim=1)   


[{'cmd': 'lmove',
  'rel': 0,
  'accel': 1200.0,
  'jerk': 2000.0,
  'cont': 0,
  'corner': 50,
  'j0': 141.7349335654358,
  'j1': 48.88754038426882,
  'j2': -79.98482008833054,
  'j3': -10.698901419291417,
  'j4': -72.36859331669143,
  'j5': -168.8100032051365,
  'j6': 390.35}]